In [1]:
import importlib
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.fft import fft, fftfreq
from pathlib import Path
from utils import data_loader_utils

importlib.reload(data_loader_utils)

<module 'utils.data_loader_utils' from '/home/scopellie/EECE5644_Project/utils/data_loader_utils.py'>

### Initial data visualization / validation of format

In [2]:
# Path for main dataset data folder
data_path = Path("./data").absolute()
# List of all h5 files in the data folder
file_list = data_loader_utils.find_all_h5s_in_dir(data_path)
print(len(file_list))
print(file_list[0])

1702
/home/scopellie/EECE5644_Project/data/M02/OP08/bad/M02_Feb_2019_OP08_000.h5


In [3]:
# Test - Reading and formatting first file
file_df = data_loader_utils.format_file(file_list[0])
file_df

,X,Y,Z,timestamp,label,machine,process,i,window
0,-7.0,-58.0,-1122.0,0.0,False,2,8,0,0
1,-17.0,101.0,-1042.0,0.5,False,2,8,0,0
2,46.0,81.0,-905.0,1.0,False,2,8,0,0
3,93.0,-35.0,-1130.0,1.5,False,2,8,0,0
4,-42.0,-74.0,-1130.0,2.0,False,2,8,0,0
...,...,...,...,...,...,...,...,...,...
63806,70.0,-50.0,-1022.0,31903.0,False,2,8,0,15
63807,81.0,-261.0,-1050.0,31903.5,False,2,8,0,15
63808,-35.0,-173.0,-1077.0,31904.0,False,2,8,0,15
63809,-42.0,-44.0,-1087.0,31904.5,False,2,8,0,15


### Extraction of data window features

In [16]:
with h5py.File(file_list[0], 'r') as f:
    dataset = f['vibration_data'][:]

dataset    

array([[   -7.,   -58., -1122.],
       [  -17.,   101., -1042.],
       [   46.,    81.,  -905.],
       ...,
       [  -35.,  -173., -1077.],
       [  -42.,   -44., -1087.],
       [   76.,   187., -1116.]], shape=(63811, 3), dtype=float32)

In [37]:
window_size = 4000
windows = []
df = pd.DataFrame()

for file in file_list[:]:
    with h5py.File(file, 'r') as f:
        dataset = f['vibration_data'][:]
        num_windows = len(dataset) // window_size
        for i in range(0, num_windows):
            w = dataset[i * window_size: (i + 1) * window_size]
            features = data_loader_utils.extract_window_features(w, file.split("/")[-1])
            df_feat = pd.DataFrame([features])
            df_feat["label"] = 1 if file.split("/")[-2] == "good" else -1
            df = pd.concat([df, df_feat], ignore_index=True)

print(df.shape)
df.head(5)

(43847, 102)


,x_mean,x_std,x_min,x_max,x_ptp,x_rms,x_abs_mean,x_skew,x_kurtosis,x_crest_factor,...,mag_spectral_spread,mag_spectral_skew,mag_spectral_kurtosis,mag_spectral_entropy,mag_spectral_rolloff_85,mag_band_power_low,mag_band_power_mid,mag_band_power_high,filename,label
0,21.73200,422.807252,-1762.0,2061.0,3823.0,423.312604,212.52000,-0.058248,5.185042,4.868742,...,225.688346,1.488865,4.846248,5.435784,349.609375,21776.748269,9952.253847,7773.528520,M02_Feb_2019_OP08_000.h5,-1
1,16.78475,342.504766,-1657.0,1627.0,3284.0,342.873031,165.28825,-0.245972,7.717127,4.832693,...,215.601851,1.569041,5.756992,4.887399,250.000000,20701.170301,17203.491439,7385.209269,M02_Feb_2019_OP08_000.h5,-1
2,20.09675,270.577334,-1631.0,1596.0,3227.0,271.288905,124.83775,-0.194596,12.089772,6.012041,...,216.278027,1.767105,6.213662,4.945196,248.046875,23398.821260,11372.750860,6181.213237,M02_Feb_2019_OP08_000.h5,-1
3,16.16850,356.810116,-1768.0,1602.0,3370.0,357.131700,172.65150,-0.146414,6.894793,4.950555,...,229.286215,1.479508,4.962537,5.237262,281.250000,21148.569796,12037.933887,8844.105332,M02_Feb_2019_OP08_000.h5,-1
4,18.97500,308.573896,-1639.0,1528.0,3167.0,309.118255,140.03850,0.007661,8.977687,5.302178,...,216.691737,1.520109,5.413658,5.157871,255.859375,16072.888302,10306.646003,5985.625453,M02_Feb_2019_OP08_000.h5,-1


In [50]:
feature_cols = [c for c in df.columns if c != "label" and c != "filename"]

Q1 = df[feature_cols].quantile(0.25)
Q3 = df[feature_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

mask = ((df[feature_cols] >= lower_bound) & (df[feature_cols] <= upper_bound)).all(axis=1)

print(f"Original shape: {df.shape}")
df = df[mask].copy()
print(f"Cleaned shape: {df.shape}")


Original shape: (43847, 102)
Cleaned shape: (43847, 102)


### Splitting training and testing buckets

In [48]:
good_file_windows = df[df["label"] == 1].drop(columns=["filename"])
bad_file_windows = df[df["label"] == -1].drop(columns=["filename"])

good_windows = good_file_windows.shape[0]
bad_windows = bad_file_windows.shape[0]
print(f"Windows with good label: {good_windows}; Windows with bad label: {bad_windows}.")

Windows with good label: 43847; Windows with bad label: 0.


In [45]:
train_ratio = 0.8 + 0.8 * (bad_windows / good_windows)
test_ratio = 1 - train_ratio
n_train = round(good_windows * train_ratio,0)
n_test_good = round(good_windows * test_ratio, 0)

print(f"""Training ratio: {n_train / (good_windows + bad_windows)}; 
{n_train} training vectors \n""")
print(f"""Testing ratio: {(n_test_good + bad_windows) / (good_windows + bad_windows)};
{n_test_good + bad_windows} testing vectors, {n_test_good} with "good" label""")

Training ratio: 0.8000091226309668; 
35078.0 training vectors 

Testing ratio: 0.19999087736903323;
8769.0 testing vectors, 7394.0 with "good" label


In [46]:
X_train = good_file_windows.sample(frac=train_ratio, random_state=42)
y_train = X_train[["label"]]
X_train = X_train.drop(columns=["label"])
X_test = df.drop(X_train.index)
X_test = X_test.drop(columns=["filename"])
y_test = X_test[["label"]]
X_test = X_test.drop(columns=["label"])

### Isolation Forest

In [21]:
from itertools import product
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

In [22]:
clf = IsolationForest(contamination='auto', random_state=42)
clf.fit(X_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [23]:
y_anom = (y_test == -1).astype(int)

test_scores = -clf.score_samples(X_test)

print("anomaly rate:", y_anom.mean())
print("ROC-AUC:", roc_auc_score(y_anom, test_scores))
print("PR-AUC :", average_precision_score(y_anom, test_scores))

anomaly rate: label    0.156802
dtype: float64
ROC-AUC: 0.7249964836353799
PR-AUC : 0.4218725398822045


In [ ]:
results = []
for max_samples, max_features in product((128, 256, 512, 1024), (0.3, 0.5, 0.7, 1.0)):
    scores = []
    m = IsolationForest(
        n_estimators=300,
        max_samples=min(max_samples, len(X_train)),
        max_features=max_features,
        random_state=42,
        n_jobs=-1,
    ).fit(X_train)
    scores.append(-m.score_samples(X_test))
    s = np.mean(scores, axis=0)
    results.append((max_samples, max_features, roc_auc_score(y_anom, s), average_precision_score(y_anom, s)))

df_IF = pd.DataFrame(results, columns=["max_samples", "max_features", "roc_auc", "pr_auc"])
print(df_IF.sort_values("pr_auc", ascending=False))

    max_samples  max_features   roc_auc    pr_auc
14         1024           0.7  0.768912  0.499112
10          512           0.7  0.755045  0.465868
13         1024           0.5  0.750567  0.465216
15         1024           1.0  0.754757  0.464372
12         1024           0.3  0.756713  0.452216
8           512           0.3  0.759237  0.452106
6           256           0.7  0.755978  0.450709
4           256           0.3  0.747969  0.435279
11          512           1.0  0.749393  0.429828
9           512           0.5  0.748100  0.428519
7           256           1.0  0.736084  0.423629
2           128           0.7  0.737461  0.418046
5           256           0.5  0.742717  0.413847
0           128           0.3  0.743150  0.412528
3           128           1.0  0.741850  0.403433
1           128           0.5  0.744782  0.390740


### One Class SVM

In [28]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import OneClassSVM

In [33]:
ocsvm = make_pipeline(
    StandardScaler(),
    OneClassSVM(kernel="rbf", gamma=0.03, nu=0.1),
)
ocsvm.fit(X_train)

test_scores_svm = -ocsvm.decision_function(X_test)

print("anomaly rate:", y_anom.mean())
print("ROC-AUC:", roc_auc_score(y_anom, test_scores_svm))
print("PR-AUC :", average_precision_score(y_anom, test_scores_svm))

anomaly rate: label    0.156802
dtype: float64
ROC-AUC: 0.8302757518380997
PR-AUC : 0.709725143780157


In [32]:
base_gamma = 1 / (X_train.shape[1] * StandardScaler().fit_transform(X_train).var())

results = []
for g_mult in (0.1, 0.3, 1, 3):
    for nu in (0.01, 0.05, 0.1):
        m = make_pipeline(StandardScaler(), OneClassSVM(gamma=base_gamma * g_mult, nu=nu)).fit(X_train)
        s = -m.decision_function(X_test)
        results.append((g_mult, nu,
                        roc_auc_score(y_anom, s),
                        average_precision_score(y_anom, s)))

print(pd.DataFrame(results, columns=["gamma_mult", "nu", "roc_auc", "pr_auc"]))

    gamma_mult    nu   roc_auc    pr_auc
0          0.1  0.01  0.711161  0.571845
1          0.1  0.05  0.716204  0.559240
2          0.1  0.10  0.720141  0.554046
3          0.3  0.01  0.713347  0.595286
4          0.3  0.05  0.720864  0.575513
5          0.3  0.10  0.725873  0.565817
6          1.0  0.01  0.784313  0.669378
7          1.0  0.05  0.774963  0.640121
8          1.0  0.10  0.767315  0.619105
9          3.0  0.01  0.832772  0.732181
10         3.0  0.05  0.831739  0.724604
11         3.0  0.10  0.830276  0.709725
